In [1]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2LMHeadModel, GPT2Model, AutoModel
from utils.misc.metric_utils import (
    compute_per_forward_pass,
    compute_on_concatenated_passes,
    metric_name_to_function,
    EvaluationMetricSpecifications
)
from utils.misc.model_dataloader_utils import model_name_to_sizes, get_model_path, get_dataloader, get_augmentation_collated_dataloader
from utils.model_definitions.TextAutoModelWrapper import TextModelSpecifications
from utils.misc.results_saving import save_results, load_results, load_results_for_model_and_revisions

device = "cuda:1"
DISABLE_TQDM = True

# Layerwise Evaluations

In [2]:
def calculate_and_save_layerwise_metrics(
    model_specs: TextModelSpecifications,
    evaluation_metric_specs: EvaluationMetricSpecifications,
):
    dataloader_kwargs = {
        "dataset_name": "wikitext",
        "split": "train",
        "num_samples": 10000,
        "max_sample_length": 512 if model_specs.model_family == "bert" else 2048,
    }

    model_path = get_model_path(model_specs.model_family, model_specs.model_size)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path, 
                                      output_hidden_states=True, 
                                      torch_dtype=torch.bfloat16,
                                      revision=model_specs.revision).to(device)
    


    if evaluation_metric_specs.evaluation_metric == 'entropy':
        dataloader = get_dataloader(tokenizer, **dataloader_kwargs)
        compute_func_kwargs = {
            'alpha': evaluation_metric_specs.alpha,
            'normalizations': evaluation_metric_specs.normalizations
        }
        forward_pass_func = compute_per_forward_pass if evaluation_metric_specs.granularity == 'sentence' else compute_on_concatenated_passes
  

    elif evaluation_metric_specs.evaluation_metric == 'curvature':
        dataloader = get_dataloader(tokenizer, **dataloader_kwargs)
        compute_func_kwargs = {
            'k': evaluation_metric_specs.curvature_k,
        }
        forward_pass_func = compute_per_forward_pass

    elif evaluation_metric_specs.evaluation_metric == 'lidar':
        dataloader_kwargs['num_augmentations_per_sample'] = 16
        dataloader = get_augmentation_collated_dataloader(tokenizer, **dataloader_kwargs)
        compute_func_kwargs = {
            'alpha': evaluation_metric_specs.alpha,
            'normalizations': evaluation_metric_specs.normalizations,
        }
        forward_pass_func = compute_on_concatenated_passes

    elif evaluation_metric_specs.evaluation_metric == 'dime':
        dataloader_kwargs['num_augmentations_per_sample'] = 2
        dataloader = get_augmentation_collated_dataloader(tokenizer, **dataloader_kwargs)
        compute_func_kwargs = {
            'alpha': evaluation_metric_specs.alpha,
            'normalizations': evaluation_metric_specs.normalizations,
        }
        forward_pass_func = compute_on_concatenated_passes

    elif evaluation_metric_specs.evaluation_metric == 'infonce':
        dataloader_kwargs['num_augmentations_per_sample'] = 2
        dataloader = get_augmentation_collated_dataloader(tokenizer, **dataloader_kwargs)
        compute_func_kwargs = {
            'temperature': 0.1,
        }
        forward_pass_func = compute_on_concatenated_passes

    compute_func = metric_name_to_function[evaluation_metric_specs.evaluation_metric]
    results = forward_pass_func(model, dataloader, compute_func, **compute_func_kwargs)
    print(results)
    save_results(results, model_specs, evaluation_metric_specs, dataloader_kwargs)


In [ ]:
from itertools import product

def is_already_computed(model_specs, evaluation_metric_specs):
    try:
        results = load_results(model_specs, evaluation_metric_specs, {'dataset_name': 'wikitext'})
        return results is not None
    except:
        return False
    

model_names = ['mamba']
#evaluation_metrics = ['infonce', 'dime', 'lidar', 'sentence-entropy', 'dataset-entropy', 'curvature']
evaluation_metrics = ['curvature']
# pythia_revision_steps = [1, 1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000]
# revisions = ['main'] + [f"step{step}" for step in pythia_revision_steps]
revisions = ['main']
num_samples = 100

for model_name, revision, evaluation_metric in product(model_names, revisions, evaluation_metrics):
    for model_size in model_name_to_sizes[model_name]:
        if model_name == 'mamba' and not model_size == '370m':
            continue
        try:
            model_spec = TextModelSpecifications(model_family=model_name, model_size=model_size, revision=revision)
            eval_spec = EvaluationMetricSpecifications(
                evaluation_metric=evaluation_metric,
                alpha=2,
                num_samples=num_samples
            )

            if evaluation_metric == 'curvature' or not is_already_computed(model_spec, eval_spec):
                print(f"Computing: {model_name}, {evaluation_metric}, {revision}, {model_size}")
                calculate_and_save_layerwise_metrics(model_spec, eval_spec)
            else:
                print(f"Already computed: {model_name}, {evaluation_metric}, {revision}, {model_size}")
            
        except Exception as e:
            print(f"\tError for {model_name}, {evaluation_metric}, {revision}: {str(e)}")
            raise e

## Data Augmentation

In [ ]:
from utils.misc.model_dataloader_utils import text_augmentation

input_text = ["The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog.The quick brown fox jumps over the lazy dog.The quick brown fox jumps over the lazy dog. "]

augmented_text = text_augmentation(input_text, num_augmentations_per_sample=10)
print(f"Original text: {input_text}")
for i, t in enumerate(augmented_text[0].split(',')):
    print(f"Augmented text {i}: {t}")

# Plotting

In [ ]:
import matplotlib.pyplot as plt
import pickle

def load_results_layerwise_entropies(model_name, experiment_name, granularity, normalization):
    with open(f"entropy_results/entropy={experiment_name}_model={model_name}_granularity={granularity}_normalization={normalization}.pkl", "rb") as f:
        layerwise_entropies_per_model = pickle.load(f)
    return layerwise_entropies_per_model

SHOULD_USE_DEPTH_PERCENTAGE =True 


models = ["EleutherAI", "mamba", "bert"] 
experiments = ["curvature", "entropy", "entropy","lidar", "dime", 'infonce']

A = len(models)
B = len(experiments)
fig, axs = plt.subplots(A, B, figsize=(5 * B, 5 * A))

row_labels = ['Pythia', 'Mamba', 'BERT']
col_labels = ['Curvature', 'Von Neumann Entropy (Prompt)', 'Von Neumann Entropy (Dataset)', 'LIDAR', 'DiME']

# Iterate over models and experiments
for i, model_family in enumerate(models):
    for j, evaluation_metric in enumerate(experiments):
        # Load entropies for each model and experiment
        granularity = 'sentence' if 'entropy' in evaluation_metric and j==1 else 'dataset'
        granularity = 'sentence' if 'curvature' in evaluation_metric else granularity

        normalization = 'maxEntropy'
        normalization = 'raw' if 'curvature' in evaluation_metric else normalization

        try:
            layerwise_entropies = load_results_layerwise_entropies(model_family, evaluation_metric, granularity=granularity, normalization=normalization)
        except FileNotFoundError as e:
            print(e)
            layerwise_entropies=None

        # Access the corresponding subplot
        ax = axs[i, j] if A > 1 and B > 1 else axs[j] if A == 1 else axs[i]

        if layerwise_entropies is not None:
            # for pythia, only plot 1B and 1.4B models
            if model_family == "EleutherAI":
                layerwise_entropies = {k: v for k, v in layerwise_entropies.items() if k not in ['1.4b', '2.8b', '6.9b']}
            for model_size_idx, (model_variant, entropies) in enumerate(layerwise_entropies.items()):
                offset = 4
                color = plt.cm.BuGn( (model_size_idx+offset) / (len(layerwise_entropies) + offset) )

                if SHOULD_USE_DEPTH_PERCENTAGE:
                    depth_percentage = np.linspace(0, 100, len(entropies))
                    ax.plot(depth_percentage, entropies, marker='o', label=model_variant, color=color)
                else:
                    ax.plot(entropies, marker='o', label=model_variant, color=color)
        else:
            ax.text(0.5, 0.5, 'Missing data', ha='center', va='center', fontsize=12, color='red')

        if SHOULD_USE_DEPTH_PERCENTAGE:
            ax.set_xlabel('Depth Percentage')
        else:   
            ax.set_xlabel('Layer')
        ax.set_ylabel(f'{col_labels[j]}')
        if i==0:
            ax.set_title(f'{col_labels[j]}')
        
        if j == 0:
            ax.legend()
        ax.grid(True)
# Set row labels using ax.text
for i, row in enumerate(row_labels):
    y_position = (A-i-0.5)/A
    if i == len(models)-1:
        y_position += 0.02
    fig.text(-0.001, y_position, row, rotation=90, va='center', ha='right', fontsize='x-large')

# Adjust layout
plt.tight_layout()
plt.show()

# Plot Normalizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_family = "Pythia"
revisions = ['main']
evaluation_metric = "sentence-entropy"
model_sizes = model_name_to_sizes[model_family]

results = load_results_for_model_and_revisions(model_family, model_sizes, revisions, [evaluation_metric])
# Create a figure with subplots for each normalization
normalizations = list(next(iter(results.values())).keys())
fig, axs = plt.subplots(1, len(normalizations), figsize=(18, 6), squeeze=False)
axs = axs.flatten()  # Flatten axs to handle both single and multiple subplots
fig.suptitle(f'{evaluation_metric.capitalize()} for {model_family.capitalize()} Models', fontsize=16)

for i, normalization in enumerate(normalizations):
    for model_size_idx, model_size in enumerate(model_sizes):
        entropies = results[(revisions[0], evaluation_metric, model_size)][normalization]
        offset = 4
        color = plt.cm.BuGn((model_size_idx + offset) / (len(model_sizes) + offset))
        
        if True:
            depth_percentage = np.linspace(0, 100, len(entropies))
            axs[i].plot(depth_percentage, entropies, marker='o', label=f'{model_size}', color=color)
        else:
            axs[i].plot(entropies, marker='o', label=f'{model_size}', color=color)
    
    axs[i].set_title(f'Normalization: {normalization.capitalize()}')
    axs[i].set_xlabel('Depth Percentage')
    axs[i].set_ylabel(evaluation_metric.capitalize())
    axs[i].legend()
    axs[i].grid(True)

plt.tight_layout()
plt.show()

# Plot Model Checkpoints

In [13]:
plt.rcParams['font.size'] = 16

In [15]:
import cmasher as cmr
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors

models = ["Pythia"]
metrics = ["sentence-entropy", 'curvature', "infonce", "lidar", 'dime']
pythia_revision_steps = [1, 1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000]
revisions = ['main'] + [f"step{step}" for step in pythia_revision_steps]

results = load_results_for_model_and_revisions("Pythia", "410m", revisions, metrics)

model_name_to_label = {
    "Pythia": "Pythia",
}
metric_name_to_label = {
    "sentence-entropy": "Prompt Entropy",
    "dataset-entropy": "Batch Entropy",
    "infonce": "InfoNCE",
    "lidar": "LiDAR",
    "dime": "DiME",
    "curvature": "Curvature",
}

metric_name_to_normalization = {
    "curvature": "raw",
    "dime": "raw",
    "sentence-entropy": "maxEntropy",
    "dataset-entropy": "raw",
    "infonce": "raw",
    "lidar": "raw"
}

# Add a new metric for DiME normalized by dataset entropy
metrics.append("dime-normalized")
metric_name_to_label["dime-normalized"] = "DiME divided by Prompt Entropy"
metric_name_to_normalization["dime-normalized"] = "raw"
print(results.keys())
for revision in revisions:
    dime_results = results[(revision, "dime")]["raw"]
    dataset_entropy_results = results[(revision, "sentence-entropy")]["maxEntropy"]
    if len(dime_results) != len(dataset_entropy_results):
        raise ValueError(f"Length mismatch for revision {revision}: DiME ({len(dime_results)}) vs Dataset Entropy ({len(dataset_entropy_results)})")
    normalized_results = [dime / entropy if entropy != 0 else 0 for dime, entropy in zip(dime_results, dataset_entropy_results)]
    results[(revision, "dime-normalized")] = {"raw": normalized_results}

# Split metrics into two rows
top_metrics = metrics[:3]
bottom_metrics = metrics[3:]

fig, axs = plt.subplots(2, 3, figsize=(18, 10))

for row, row_metrics in enumerate([top_metrics, bottom_metrics]):
    for col, evaluation_metric in enumerate(row_metrics):
        ax = axs[row, col]
        
        if results:
            cmap = plt.cm.Greens
            norm = colors.LogNorm(vmin=min(pythia_revision_steps), vmax=max(pythia_revision_steps))

            for revision in revisions:
                metric_results = results[(revision, evaluation_metric)]
                metric_results = metric_results[metric_name_to_normalization[evaluation_metric]]
                
                if revision == 'main':
                    step = 143000
                else:
                    step = int(revision.split('step')[1])
                if step < 10:
                    color = cmap(norm(step) + 0.2)
                else:
                    color = cmap(norm(step))

                ax.plot(range(len(metric_results)), metric_results, marker='o', color=color, label=f'Step {step}')
        else:
            ax.text(0.5, 0.5, 'Missing data', ha='center', va='center', fontsize=12, color='red')

        ax.set_xlabel('Layer')
        ax.set_ylabel(metric_name_to_label[evaluation_metric])
        ax.set_title(metric_name_to_label[evaluation_metric])
        
        ax.grid(True)

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax)
        cbar.set_label('Training Step')

plt.tight_layout()
plt.savefig(f"figures/metrics_at_pythia_checkpoints.pdf")
plt.show()


# Results Across Architectures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors

metrics = ["sentence-entropy", 'curvature', "infonce", "lidar", 'dime']
model_families = ["Pythia", "mamba"]

family_to_model_size = {
    "Pythia": "410m",
    "mamba": "370m",
}

family_to_results = {}
for model_family in model_families:
    family_to_results[model_family] = load_results_for_model_and_revisions(model_family, family_to_model_size[model_family], ["main"], metrics)

metric_name_to_label = {
    "sentence-entropy": "Normalized Prompt Entropy",
    "dataset-entropy": "Batch Entropy",
    "infonce": "InfoNCE",
    "lidar": "LiDAR",
    "dime": "DiME",
    "curvature": "Curvature",
}

metric_name_to_normalization = {
    "curvature": "raw",
    "dime": "raw",
    "sentence-entropy": "maxEntropy",
    "dataset-entropy": "raw",
    "infonce": "raw",
    "lidar": "raw"
}

# Add a new metric for DiME normalized by dataset entropy
metrics.append("dime-normalized")
metric_name_to_label["dime-normalized"] = "DiME divided by Prompt Entropy"
metric_name_to_normalization["dime-normalized"] = "raw"

for model_family in model_families:
    results = family_to_results[model_family]
    for revision in ["main"]:
        dime_results = results[(revision, "dime")]["raw"]
        dataset_entropy_results = results[(revision, "sentence-entropy")]["maxEntropy"]
        if len(dime_results) != len(dataset_entropy_results):
            raise ValueError(f"Length mismatch for {model_family} revision {revision}: DiME ({len(dime_results)}) vs Dataset Entropy ({len(dataset_entropy_results)})")
        normalized_results = [dime / entropy if entropy != 0 else 0 for dime, entropy in zip(dime_results, dataset_entropy_results)]
        results[(revision, "dime-normalized")] = {"raw": normalized_results}

# Split metrics into two rows
top_metrics = metrics[:3]
bottom_metrics = metrics[3:]

fig, axs = plt.subplots(2, 3, figsize=(16, 10))

for row, row_metrics in enumerate([top_metrics, bottom_metrics]):
    for col, evaluation_metric in enumerate(row_metrics):
        ax = axs[row, col]
        
        for model_family in model_families:
            results = family_to_results[model_family]
            metric_results = results[("main", evaluation_metric)]
            metric_results = metric_results[metric_name_to_normalization[evaluation_metric]]
            depth_percentage = np.linspace(0, 100, len(metric_results))
            color = 'green' if model_family == "Pythia" else 'blue'
            ax.plot(depth_percentage, metric_results, marker='o', color=color, label=model_family)

        ax.set_xlabel('Depth Percentage')
        ax.set_ylabel(metric_name_to_label[evaluation_metric])
        ax.set_title(metric_name_to_label[evaluation_metric])
        
        ax.grid(True)
        if row == 0 and col == 0:
            ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(f"figures/metrics_comparison_pythia_mamba_llama.pdf")
plt.show()


# Plot all metrics for different model sizes

In [ ]:
import matplotlib

# Load results for all Pythia model sizes
model_family = "Pythia"
model_sizes = ["14m", "70m", "160m", "410m", "1b"]
metrics = ["sentence-entropy", "curvature", "infonce", "lidar", "dime", "dime-normalized"]

fig, axs = plt.subplots(2, 3, figsize=(18, 12))
axs = axs.flatten()

for idx, metric in enumerate(metrics):
    ax = axs[idx]
    
    for model_idx, size in enumerate(model_sizes):
        model_specs = ModelSpecifications(model_family, size, "main")
        if metric == "dime-normalized":
            metric = "dime"
            evaluation_metric_specs = EvaluationMetricSpecifications(metric)
            dataloader_kwargs = {'dataset_name': 'wikitext'}
            dime_results = load_results(model_specs, evaluation_metric_specs, dataloader_kwargs)
            dime_results = dime_results["raw"]
            dataset_entropy_results = load_results_for_model_and_revisions(model_family, size, ["main"], ["sentence-entropy"])[("main", "sentence-entropy")]["maxEntropy"]
            normalized_results = [dime / entropy if entropy != 0 else 0 for dime, entropy in zip(dime_results, dataset_entropy_results)]
            values = normalized_results
        else:
            evaluation_metric_specs = EvaluationMetricSpecifications(metric)
            dataloader_kwargs = {'dataset_name': 'wikitext'}
        
            results = load_results(model_specs, evaluation_metric_specs, dataloader_kwargs)
        
        if results is not None:
            normalization = "raw" if metric in ["dime", "infonce", "curvature"] else "maxEntropy"
            values = results[normalization]
            color = plt.cm.get_cmap('Greens')(np.linspace(0.5, 1, len(model_sizes)))
            if SHOULD_USE_DEPTH_PERCENTAGE:
                x = np.linspace(0, 100, len(values))
                ax.plot(x, values, label=f"{size}", marker='o', color=color[model_idx])
                ax.set_xlabel("Depth Percentage")
            else:
                ax.plot(values, label=f"{size}", marker='o', color=color[model_idx])
                ax.set_xlabel("Layer")
        
    ax.set_ylabel(metric_name_to_label[metric])
    ax.set_title(metric_name_to_label[metric])
    if idx == 0:
        ax.legend(title="Model Size", loc='lower left')
    ax.grid(True)

plt.tight_layout()
plt.savefig(f"figures/all_metrics_{model_family}_sizes.pdf", bbox_inches='tight')
plt.show()


# Todo: for dime

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors

SHOULD_USE_DEPTH_PERCENTAGE = False

models = ["EleutherAI"]
experiments = ["dime"]

A = len(models)
B = len(experiments)
fig, axs = plt.subplots(A, B, figsize=(5 * B, 5 * A))
if A == 1:
    axs = [axs]


dime_results = load_results_layerwise_entropies("EleutherAI", "dime", granularity="sentence", normalization="raw")

row_labels = ['Pythia']
col_labels = ['DiME']

# Iterate over models and experiments
for i, model_family in enumerate(models):
    for j, evaluation_metric in enumerate(experiments):
        # Load entropies for each model and experiment
        granularity = 'dataset'

        normalization = 'raw'
        normalization = 'raw' if 'curvature' in evaluation_metric else normalization
        normalization = 'raw' if 'infonce' in evaluation_metric else normalization
        normalization = 'maxEntropy' if 'lidar' in evaluation_metric else normalization

        layerwise_entropies = load_results_layerwise_entropies(model_family, evaluation_metric, granularity=granularity, normalization=normalization, revisioned=True)

        # Access the corresponding subplot
        ax = axs[i, j] if A > 1 and B > 1 else axs[j] if A == 1 else axs[i]
        if layerwise_entropies is not None:
            for model_size_idx, (model_variant, entropies) in enumerate(layerwise_entropies.items()):
                step = int(model_variant.split('_step')[1])
                color = plt.cm.Greens((30000+step) / 150000)  # Normalize step to [0, 1]


                entropies /= (2*entropy_results)**1

                if SHOULD_USE_DEPTH_PERCENTAGE:
                    depth_percentage = np.linspace(0, 100, len(entropies))
                    ax.plot(depth_percentage, entropies, marker='o', color=color)
                else:
                    ax.plot(entropies, marker='o', color=color)
        else:
            ax.text(0.5, 0.5, 'Missing data', ha='center', va='center', fontsize=12, color='red')

        if SHOULD_USE_DEPTH_PERCENTAGE:
            ax.set_xlabel('Depth Percentage')
        else:   
            ax.set_xlabel('Layer')
        ax.set_ylabel(f'{col_labels[j]}')
        if i==0:
            ax.set_title(f'{col_labels[j]}')
        
        ax.grid(True)

        # Add colorbar as legend
        cmap = plt.cm.get_cmap('Greens').copy()
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=140000))
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax)
        cbar.set_label('Training Step')

# Set row labels using ax.text
for i, row in enumerate(row_labels):
    y_position = (A-i-0.5)/A
    if i == len(models)-1:
        y_position += 0.02
    fig.text(-0.001, y_position, row, rotation=90, va='center', ha='right', fontsize='x-large')

# Adjust layout
plt.tight_layout()
plt.show()